In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.dataset import DownscaleDataset

In [3]:
from src.dataset import DownscaleDataset

train_ds = DownscaleDataset("../.data/downscaling_splits/train_norm.nc")
x, y, mask = train_ds[0]

print(x.shape)     # should be [2, H, W]
print(y.shape)     # should be [1, H, W]
print(mask.shape)  # should be [1, H, W]
print(x.dtype, y.dtype, mask.dtype)

torch.Size([2, 240, 311])
torch.Size([1, 240, 311])
torch.Size([1, 240, 311])
torch.float32 torch.float32 torch.float32


In [4]:
from src.model_cnn import SimpleCNN

model = SimpleCNN()
out = model(x.unsqueeze(0))   # add batch dimension

print("input shape :", x.unsqueeze(0).shape)
print("output shape:", out.shape)
print("target shape:", y.unsqueeze(0).shape)

input shape : torch.Size([1, 2, 240, 311])
output shape: torch.Size([1, 1, 240, 311])
target shape: torch.Size([1, 1, 240, 311])


In [5]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)

xb, yb, mb = next(iter(train_loader))
print(xb.shape, yb.shape, mb.shape)

torch.Size([4, 2, 240, 311]) torch.Size([4, 1, 240, 311]) torch.Size([4, 1, 240, 311])


In [6]:
import torch
from src.model_cnn import SimpleCNN

model = SimpleCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

xb, yb, mb = next(iter(train_loader))
pred = model(xb)

loss = ((pred - yb) ** 2 * mb).sum() / mb.sum().clamp_min(1.0)
print(loss.item())

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("One training step passed.")

0.9320708513259888
One training step passed.


In [13]:
print("xb has nan:", torch.isnan(xb).any().item())
print("yb has nan:", torch.isnan(yb).any().item())
print("mb has nan:", torch.isnan(mb).any().item())
print("pred has nan:", torch.isnan(pred).any().item())

print("xb nan count:", torch.isnan(xb).sum().item())
print("yb nan count:", torch.isnan(yb).sum().item())
print("mb nan count:", torch.isnan(mb).sum().item())
print("pred nan count:", torch.isnan(pred).sum().item())

xb has nan: False
yb has nan: False
mb has nan: False
pred has nan: False
xb nan count: 0
yb nan count: 0
mb nan count: 0
pred nan count: 0


In [14]:
print("xb has inf:", torch.isinf(xb).any().item())
print("yb has inf:", torch.isinf(yb).any().item())
print("pred has inf:", torch.isinf(pred).any().item())

xb has inf: False
yb has inf: False
pred has inf: False
